# SCX in five minutes — Python

**Run these cells from top to bottom in your existing Scanpy/AnnData Python kernel.**
This notebook creates its own tiny data: no H5AD download or SCX installation is needed.
It downloads the pinned **alpha.7** script and checks SHA-256 *before* executing it.
Nothing installs or upgrades packages. The saved outputs show what success looks like.

**You will:** make a small object → write SCX → read it back → check values and metadata →
view a regular TSV → select actual cells/genes. The [matching R lesson](detailed.R)
uses the same expression values and IDs without needing Jupyter or Python.

Use **Kernel → Restart Kernel and Run All** for a clean start. A complete rerun creates
a new output folder and preserves older results. Rerunning only a write cell will
deliberately trigger overwrite protection; rerun step 2 first to choose a new folder.
The first run needs internet; verified cached files work offline thereafter.


## 1. Check the current environment
Choose the kernel that already contains AnnData (usually your Scanpy environment).
If a dependency is missing, switch kernels/environments. This lesson does not run pip.


In [2]:
import importlib.util
import sys

if sys.version_info < (3, 10):
    raise RuntimeError("Choose your Python 3.10+ Scanpy/AnnData kernel.")
needed = ["anndata", "numpy", "pandas", "scipy"]
missing = [name for name in needed if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError("Switch to your analysis kernel; missing: " + ", ".join(missing))

import anndata as ad
import numpy as np
import pandas as pd
from scipy import sparse

print("Ready: existing AnnData / NumPy / pandas / SciPy environment. No packages installed.")


Ready: existing AnnData / NumPy / pandas / SciPy environment. No packages installed.


## 2. Fetch and verify the fixed file; create fresh outputs
The hashes below are pinned to the published **v0.1.0-alpha.7** assets, including
the checksum list itself. Existing cached files are verified, never silently replaced.
If the network is unavailable, place `scx.py` and `SHA256SUMS` from that release in
`scx-tutorial/files/` under the current working folder and rerun this cell.
Outputs go into a new `scx-tutorial/output/run-*` folder each time. No files are deleted.


In [3]:
from pathlib import Path
import hashlib
import tempfile
import time
import urllib.request

RELEASE = "v0.1.0-alpha.7"
BASE_URL = f"https://github.com/junha-shin/scx-format/releases/download/{RELEASE}/"
EXPECTED = {
    "scx.py": "3384f730e621c2ede7110da6b9e9c39bafb578fa886e8a9b752751c692b7fc92",
    "SHA256SUMS": "f7c277c380d04ed1f2766b0c9726d6b32c69811cf51a7b426808efb9cadfe914",
}
HOME = Path.cwd() / "scx-tutorial"
FILES = HOME / "files"
FILES.mkdir(parents=True, exist_ok=True)

def verified_file(name):
    path = FILES / name
    if path.exists():
        payload = path.read_bytes()
    else:
        for attempt in range(2):
            try:
                with urllib.request.urlopen(BASE_URL + name, timeout=20) as response:
                    payload = response.read()
                break
            except OSError as error:
                if attempt == 1:
                    raise RuntimeError("Download failed. Copy the alpha.7 files into scx-tutorial/files/ for offline use.") from error
                time.sleep(0.5)
    if hashlib.sha256(payload).hexdigest() != EXPECTED[name]:
        raise RuntimeError(f"Checksum mismatch: {name}. Inspect the cached file and download the pinned release again; it was not executed or overwritten.")
    if not path.exists():
        with path.open("xb") as stream:
            stream.write(payload)
    return path.resolve()

checksum_path = verified_file("SHA256SUMS")
SCRIPT_PATH = verified_file("scx.py")
assert EXPECTED["scx.py"] + "  scx.py" in checksum_path.read_text(encoding="utf-8").splitlines()

output_root = HOME / "output"
output_root.mkdir(exist_ok=True)
RUN_DIR = Path(tempfile.mkdtemp(prefix="run-", dir=output_root))
print("Verified alpha.7 bytes. Created a fresh run-* folder under scx-tutorial/output/.")


Verified alpha.7 bytes. Created a fresh run-* folder under scx-tutorial/output/.


## 3. Load that verified file
We load the saved file explicitly, so an installed package named `scx` cannot be
picked by accident. The variable is still named `scx`, so the ordinary API is unchanged.


In [4]:
loaded = sys.modules.get("_scx_standalone")
if loaded is not None and not str(getattr(loaded, "__file__", "")).startswith(str(SCRIPT_PATH) + "!/"):
    raise RuntimeError("Another standalone SCX is already loaded. Restart the kernel and Run All.")
spec = importlib.util.spec_from_file_location("scx_tutorial_verified", SCRIPT_PATH)
scx = importlib.util.module_from_spec(spec)
spec.loader.exec_module(scx)
assert scx.SCX_VERSION == RELEASE
print("Loaded", scx.SCX_VERSION, "from the verified file.")


Loaded v0.1.0-alpha.7 from the verified file.


## 4. Create tiny data you can inspect
Rows are **cells**, columns are **features**. This small display uses a dense table
only because it has 12 values; do not densify your real dataset for SCX.
For this transport lesson, normalized values are simply `log1p(counts)`; no biological
analysis/preprocessing is being performed. Numeric, boolean and ordered-category
metadata include missing values and an unused category.


In [5]:
cell_ids = ["cell1", "cell2", "cell3"]
feature_ids = ["geneA", "geneB", "geneC", "geneD"]
counts = sparse.csr_matrix([[1, 0, 3, 0], [0, 2, 0, 4], [5, 1, 0, 0]], dtype=np.int32)
normalized = counts.astype(np.float64)
normalized.data = np.log1p(normalized.data)
obs = pd.DataFrame(index=cell_ids)
obs["sample_number"] = pd.array([1, None, 3], dtype="Int32")
obs["flag"] = pd.array([True, None, False], dtype="boolean")
obs["group"] = pd.Categorical(["B", "A", "B"], categories=["B", "A", "unused"], ordered=True)
var = pd.DataFrame({"symbol": pd.Series(["A", "B", "C", "D"], index=feature_ids)})
demo = ad.AnnData(X=normalized, obs=obs, var=var)
demo.layers["counts"] = counts

print("Expected: 3 cells × 4 features")
display(pd.DataFrame(counts.toarray(), index=cell_ids, columns=feature_ids))
display(demo.obs)


Expected: 3 cells × 4 features


,geneA,geneB,geneC,geneD
cell1,1,0,3,0
cell2,0,2,0,4
cell3,5,1,0,0


,sample_number,flag,group
cell1,1,True,B
cell2,<NA>,<NA>,A
cell3,3,False,B


## 5. Write SCX, read it back, and check success
The default takes counts from `layers['counts']` and normalized expression from `X`.
Validation is automatic. This demo has no embeddings/graphs; use `analysis=True` when
you intentionally want supported analysis coordinates/graphs in a real export.
Success means counts match exactly, normalized values agree to `1e-12`, and both
metadata tables retain their types, values, missing values and category order.


In [6]:
scx_dir = RUN_DIR / "demo.scx"
scx.write_scx(demo, scx_dir)
restored = scx.read_scx(scx_dir)
np.testing.assert_array_equal(restored.layers["counts"].toarray(), counts.toarray())
normalized_error = float(np.max(np.abs(restored.X.toarray() - normalized.toarray())))
assert normalized_error <= 1e-12
pd.testing.assert_frame_equal(demo.obs, restored.obs, check_exact=True)
pd.testing.assert_frame_equal(demo.var, restored.var, check_exact=True)
assert restored.shape == (3, 4)
print("PASS: counts, normalized expression, both axes and typed metadata preserved.")
display(restored.obs)


PASS: counts, normalized expression, both axes and typed metadata preserved.


,sample_number,flag,group
cell1,1,True,B
cell2,<NA>,<NA>,A
cell3,3,False,B


## 6. View ordinary TSV output
The first column is `cell_id`; all selected feature columns, including zeros,
are written in CELL × FEATURE orientation. `.gz` on the filename enables compression.
These two tables should match the counts table and `log1p` values above.


In [7]:
counts_tsv = RUN_DIR / "counts.tsv"
normalized_tsv = RUN_DIR / "normalized.tsv.gz"
scx.to_tsv(scx_dir, counts_tsv)
scx.to_tsv(scx_dir, normalized_tsv, matrix="normalized")
counts_table = pd.read_csv(counts_tsv, sep="\t", index_col="cell_id")
normalized_table = pd.read_csv(normalized_tsv, sep="\t", index_col="cell_id")
np.testing.assert_array_equal(counts_table.to_numpy(), counts.toarray())
np.testing.assert_allclose(normalized_table.to_numpy(), normalized.toarray(), rtol=0, atol=1e-12)
display(counts_table)
display(normalized_table)


,geneA,geneB,geneC,geneD
cell_id,,,,
cell1,1,0,3,0
cell2,0,2,0,4
cell3,5,1,0,0


,geneA,geneB,geneC,geneD
cell_id,,,,
cell1,0.693147,0.000000,1.386294,0.000000
cell2,0.000000,1.098612,0.000000,1.609438
cell3,1.791759,0.693147,0.000000,0.000000


## 7. Select real IDs and preserve the requested order
These IDs exist in this demo. The resulting table is **cell3, cell1** by **geneB,
geneA**, with values `[[1, 5], [0, 1]]`. The exporter accepts ID lists, not numeric
positions; this keeps the meaning the same in Python and R.


In [8]:
selected_cells = ["cell3", "cell1"]
selected_features = ["geneB", "geneA"]
subset_tsv = RUN_DIR / "subset.tsv.gz"
scx.to_tsv(scx_dir, subset_tsv, cells=selected_cells, features=selected_features)
subset_table = pd.read_csv(subset_tsv, sep="\t", index_col="cell_id")
assert subset_table.index.tolist() == selected_cells
assert subset_table.columns.tolist() == selected_features
np.testing.assert_array_equal(subset_table.to_numpy(), [[1, 5], [0, 1]])
display(subset_table)


,geneB,geneA
cell_id,,
cell3,1,5
cell1,0,1


## 8. See overwrite protection safely
This intentionally tries the existing counts TSV and catches the expected refusal.
The original bytes must remain unchanged. Choose a fresh filename or rerun step 2;
the lesson never deletes old outputs to make a rerun work.


In [9]:
previous_hash = hashlib.sha256(counts_tsv.read_bytes()).hexdigest()
try:
    scx.to_tsv(scx_dir, counts_tsv)
except FileExistsError:
    print("Expected refusal: counts.tsv already exists; its bytes are preserved.")
else:
    raise AssertionError("Overwrite protection did not refuse the existing file")
assert hashlib.sha256(counts_tsv.read_bytes()).hexdigest() == previous_hash


Expected refusal: counts.tsv already exists; its bytes are preserved.


## 9. Optional: use your own H5AD
Leave the switch **False** during the first Run All. When ready, set your file path
and make an explicit counts choice. `layers['counts']` must contain raw counts.
Only if `X` really contains raw counts, choose `COUNTS_SOURCE="X"` and
`NORMALIZED_SOURCE=None`. `raw.X` is not automatically assumed to be counts.
This reads the original H5AD without changing it, writes into the fresh run folder,
and exports only a small real-ID preview. Keep the original analysis object.


In [10]:
USE_OWN_DATA = False
INPUT_H5AD = Path("my-data.h5ad")
COUNTS_SOURCE = "counts"
NORMALIZED_SOURCE = "X"

if USE_OWN_DATA:
    own = ad.read_h5ad(INPUT_H5AD)
    print("Your object:", own.shape, "layers:", list(own.layers.keys()))
    own_scx = RUN_DIR / "own.scx"
    scx.write_scx(own, own_scx, counts=COUNTS_SOURCE, normalized=NORMALIZED_SOURCE)
    own_restored = scx.read_scx(own_scx)
    scx.to_tsv(own_scx, RUN_DIR / "own-preview.tsv.gz",
               cells=own_restored.obs_names[:10].tolist(),
               features=own_restored.var_names[:10].tolist())
else:
    print("Own-data example skipped. Change the switch only when your path/counts choices are ready.")


Own-data example skipped. Change the switch only when your path/counts choices are ready.


## Success — what was created?
`demo.scx/` contains the manifest, typed metadata and sparse matrices. `counts.tsv`,
`normalized.tsv.gz` and `subset.tsv.gz` are ordinary expression views. Existing runs
remain under `scx-tutorial/output/`. TSV is dense on disk and can be large on real
data; use subsets and gzip. Input validation still needs sparse-matrix memory.

For the other language, use the matching [R lesson](detailed.R). R standalone
requires a new output directory and refuses `overwrite=TRUE`; both languages refuse
existing TSV files. [Detailed guide](../docs/tsv-export.md) ·
[PBMC3k real-data verification](../examples/tsv-pbmc3k/README.md).


In [11]:
tutorial_result = {
    "passed": True, "SCX_VERSION": scx.SCX_VERSION, "shape": list(restored.shape),
    "counts_equal": True, "normalized_max_absolute_error": normalized_error,
    "typed_metadata_equal": True, "TSV_equal": True, "subset_order_equal": True,
    "existing_output_preserved": True,
}
print("SUCCESS: all demo checks passed. No SCX package was installed.")
print("Files in this run:")
for item in sorted(RUN_DIR.rglob("*")):
    if item.is_file():
        print("  " + item.relative_to(RUN_DIR).as_posix())


SUCCESS: all demo checks passed. No SCX package was installed.
Files in this run:
  counts.tsv
  demo.scx/cells.tsv.gz
  demo.scx/features.tsv.gz
  demo.scx/manifest.tsv
  demo.scx/matrices/counts.mtx.gz
  demo.scx/matrices/normalized.mtx.gz
  demo.scx/metadata.schema.json
  normalized.tsv.gz
  subset.tsv.gz
